 # 01 · Ingesta Bronze Batch — YelpConnect

 **Tipo:** Pipeline de ingesta  
 **Modalidad:** Batch / bootstrap controlado  
 **Entidades:** Business, User, Tip, Check-in  
 **Origen:** `/Volumes/workspace/yelp_bronze/raw`  
 **Destino:** `workspace.yelp_bronze`

 ---

 ## Propósito

 Ingerir las entidades RAW de referencia hacia Bronze preservando
 fidelidad al origen e incorporando únicamente metadata técnica.

 ## Principio Bronze

 Este notebook no limpia, no deduplica, no aplica reglas de negocio,
 no explota estructuras anidadas y no transforma semánticamente los datos.

 La carga utiliza `overwrite` de forma deliberada porque estas cuatro
 entidades se manejan como bootstrap académico completo.

 **Grano:** el mismo que el archivo RAW correspondiente.

In [0]:
import os
import sys
import json
import uuid
import importlib.util

from pyspark.sql import functions as F


# ============================================================
# 1. CARGAR MÓDULO COMÚN BRONZE
# ============================================================

COMMON_MODULE_NAME = "bronze_common"

COMMON_MODULE_PATH = os.path.join(
    os.getcwd(),
    "bronze_common.py"
)


print("=" * 72)
print("YELPCONNECT - CARGA DE CONFIGURACIÓN BRONZE")
print("=" * 72)

print(
    f"Working directory : {os.getcwd()}"
)

print(
    f"Módulo común      : {COMMON_MODULE_PATH}"
)


if not os.path.isfile(
    COMMON_MODULE_PATH
):

    raise FileNotFoundError(
        f"""
        ❌ No se encontró el módulo común Bronze.

        Ruta esperada:
        {COMMON_MODULE_PATH}
        """
    )


# ------------------------------------------------------------
# Crear loader explícito
# ------------------------------------------------------------

spec = importlib.util.spec_from_file_location(
    COMMON_MODULE_NAME,
    COMMON_MODULE_PATH
)


if (
    spec is None
    or spec.loader is None
):

    raise RuntimeError(
        f"""
        ❌ No fue posible crear el loader para:

        {COMMON_MODULE_PATH}
        """
    )


# ------------------------------------------------------------
# Crear módulo
# ------------------------------------------------------------

bronze_common = (
    importlib.util.module_from_spec(
        spec
    )
)


# ------------------------------------------------------------
# Registrar módulo en la sesión
# ------------------------------------------------------------

sys.modules[
    COMMON_MODULE_NAME
] = bronze_common


# ------------------------------------------------------------
# Ejecutar módulo
# ------------------------------------------------------------

spec.loader.exec_module(
    bronze_common
)


print(
    "✔ bronze_common.py cargado correctamente."
)


# ============================================================
# 2. EXPONER CONFIGURACIÓN COMPARTIDA
# ============================================================

CATALOG = (
    bronze_common.CATALOG
)

BRONZE_SCHEMA = (
    bronze_common.BRONZE_SCHEMA
)

RAW = (
    bronze_common.RAW
)

SOURCE_SYSTEM = (
    bronze_common.SOURCE_SYSTEM
)

LAYER = (
    bronze_common.LAYER
)

METADATA_COLUMNS = (
    bronze_common.METADATA_COLUMNS
)


# ============================================================
# 3. EXPONER FUNCIONES COMPARTIDAS
# ============================================================

assert_bronze_schema_exists = (
    bronze_common
    .assert_bronze_schema_exists
)

add_bronze_metadata = (
    bronze_common
    .add_bronze_metadata
)

apply_bronze_uc_metadata = (
    bronze_common
    .apply_bronze_uc_metadata
)

apply_bronze_metadata_column_comments = (
    bronze_common
    .apply_bronze_metadata_column_comments
)

validate_bronze_metadata = (
    bronze_common
    .validate_bronze_metadata
)

# ============================================================
# 4. CONFIGURACIÓN DEL PIPELINE BATCH
# ============================================================

PIPELINE_NAME = (
    "E7_01_bronze_batch"
)

EXECUTION_ID = str(
    uuid.uuid4()
)


ENTITIES = [
    "business",
    "user",
    "tip",
    "checkin"
]


# ============================================================
# 5. RESUMEN DE CONFIGURACIÓN
# ============================================================

print()
print("=" * 72)
print("YELPCONNECT - INGESTA BRONZE BATCH")
print("=" * 72)

print(
    f"Pipeline       : {PIPELINE_NAME}"
)

print(
    f"Execution ID   : {EXECUTION_ID}"
)

print(
    f"Catalog        : {CATALOG}"
)

print(
    f"Schema Bronze  : {BRONZE_SCHEMA}"
)

print(
    f"RAW Volume     : {RAW}"
)

print(
    f"Source System  : {SOURCE_SYSTEM}"
)

print(
    f"Layer          : {LAYER}"
)

print(
    f"Entidades      : {', '.join(ENTITIES)}"
)

print("=" * 72)

 ## 1 · Precondición

 `workspace.yelp_bronze` debe existir previamente.
 Su creación pertenece a `00_setup/01_configurar_lakehouse`.


In [0]:
# COMMAND ----------

full_schema = assert_bronze_schema_exists(
    spark
)

print(
    f"✔ Schema disponible: {full_schema}"
)

# COMMAND ----------

 ## 2 · Carga Batch

 Para cada entidad:

 1. se lee el JSON RAW en modo `PERMISSIVE`;
 2. se identifican registros corruptos;
 3. se agrega metadata técnica Bronze;
 4. se persiste en Delta;
 5. se reconcilia RAW vs Bronze.

In [0]:
# COMMAND ----------

results = []

for entity in ENTITIES:

    source_path = (
        f"{RAW}/"
        f"{entity}.json.gz"
    )

    target_table = (
        f"{CATALOG}."
        f"{BRONZE_SCHEMA}."
        f"brz_{entity}"
    )

    print()
    print("=" * 72)
    print(f"ENTIDAD : {entity}")
    print(f"ORIGEN  : {source_path}")
    print(f"DESTINO : {target_table}")
    print("=" * 72)


    # ========================================================
    # 1. LEER RAW
    # ========================================================

    try:

        df_raw = (
            spark.read
            .format("json")
            .option(
                "mode",
                "PERMISSIVE"
            )
            .option(
                "columnNameOfCorruptRecord",
                "_corrupt_record"
            )
            .load(
                source_path
            )
        )

    except Exception as exc:

        raise RuntimeError(
            f"""
            ❌ ERROR LEYENDO RAW

            Entidad:
            {entity}

            Archivo:
            {source_path}
            """
        ) from exc


    # ========================================================
    # 2. VALIDACIONES MÍNIMAS RAW
    # ========================================================

    raw_count = (
        df_raw.count()
    )

    column_count = len(
        df_raw.columns
    )


    if raw_count == 0:

        raise RuntimeError(
            f"{entity}: archivo RAW vacío."
        )


    if column_count == 0:

        raise RuntimeError(
            f"{entity}: no se detectaron columnas."
        )


    # ========================================================
    # 3. REGISTROS CORRUPTOS
    # ========================================================

    corrupt_count = 0


    if (
        "_corrupt_record"
        in df_raw.columns
    ):

        corrupt_count = (
            df_raw
            .filter(
                F.col(
                    "_corrupt_record"
                ).isNotNull()
            )
            .count()
        )


    print(
        f"RAW registros : {raw_count:,}"
    )

    print(
        f"RAW columnas  : {column_count}"
    )

    print(
        f"RAW corruptos : {corrupt_count:,}"
    )


    # ========================================================
    # 4. AGREGAR METADATA TÉCNICA BRONZE
    # ========================================================

    df_bronze = (
        add_bronze_metadata(
            df=df_raw,
            entity=entity,
            execution_id=EXECUTION_ID,
            pipeline_name=PIPELINE_NAME,
            source_file_expression=(
                F.col(
                    "_metadata.file_path"
                )
            )
        )
    )


    # ========================================================
    # 5. ESCRIBIR DELTA BRONZE
    # ========================================================

    try:

        (
            df_bronze.write
            .format(
                "delta"
            )
            .mode(
                "overwrite"
            )
            .option(
                "overwriteSchema",
                "true"
            )
            .saveAsTable(
                target_table
            )
        )

    except Exception as exc:

        raise RuntimeError(
            f"""
            ❌ ERROR ESCRIBIENDO BRONZE

            Entidad:
            {entity}

            Tabla:
            {target_table}
            """
        ) from exc


    # ========================================================
    # 6. APLICAR METADATA UNITY CATALOG
    # ========================================================

    apply_bronze_uc_metadata(
        spark=spark,
        table_name=target_table,
        entity=entity,
        description=(
            f"Capa Bronze de {entity} "
            f"del Yelp Open Dataset. "
            f"Conserva fidelidad al origen "
            f"e incorpora metadata técnica "
            f"de ingesta, trazabilidad, "
            f"schema y checksum."
        ),
        criticality="medium",
        classification="internal",
        retention_class="raw_source"
    )


    # ========================================================
    # 7. COMENTARIOS DE COLUMNAS TÉCNICAS
    # ========================================================

    apply_bronze_metadata_column_comments(
        spark=spark,
        table_name=target_table
    )


    # ========================================================
    # 8. VALIDAR METADATA TÉCNICA
    # ========================================================

    metadata_validation = (
        validate_bronze_metadata(
            spark.table(
                target_table
            )
        )
    )


    if not metadata_validation[
        "metadata_ok"
    ]:

        raise RuntimeError(
            f"""
            ❌ METADATA BRONZE INCOMPLETA

            Entidad:
            {entity}

            Tabla:
            {target_table}

            Columnas faltantes:
            {
                metadata_validation[
                    "missing_columns"
                ]
            }
            """
        )


    # ========================================================
    # 9. RECONCILIACIÓN RAW → BRONZE
    # ========================================================

    bronze_count = (
        spark
        .table(
            target_table
        )
        .count()
    )


    if (
        raw_count
        !=
        bronze_count
    ):

        raise RuntimeError(
            f"""
            ❌ ERROR DE RECONCILIACIÓN

            Entidad:
            {entity}

            RAW:
            {raw_count}

            BRONZE:
            {bronze_count}
            """
        )


    # ========================================================
    # 10. REGISTRAR RESULTADO
    # ========================================================

    results.append({

        "entity":
            entity,

        "raw_records":
            raw_count,

        "bronze_records":
            bronze_count,

        "corrupt_records":
            corrupt_count,

        "metadata_ok":
            metadata_validation[
                "metadata_ok"
            ],

        "target_table":
            target_table
    })


    print()
    print(
        f"✔ {entity}: "
        f"{bronze_count:,} filas "
        f"| corruptos={corrupt_count:,} "
        f"| metadata=OK"
    )

In [0]:
# COMMAND ----------

display(
    spark.createDataFrame(results)
    .orderBy("entity")
)

summary = {
    "status": "SUCCESS",
    "pipeline": PIPELINE_NAME,
    "execution_id": EXECUTION_ID,
    "entities": len(results),
    "rows_bronze": sum(
        item["bronze_records"]
        for item in results
    )
}

dbutils.notebook.exit(json.dumps(summary))